In [ ]:
import pandas as pd
import numpy as np
import os
import datetime
# import kaggle
# from kaggle.api.kaggle_api_extended import KaggleApi
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pickle 

In [9]:

DATASET = "hiranomanabu/ransmap-2024-ransomware-behavioral-features"

strg_access_column_names = ['unix_time_s', 'unix_time_ns', 'lba', 'storage_size', 'storage_entropy']
mem_access_columns_names = ['unix_time_s', 'unix_time_ns', 'gpa', 'mem_size', 'mem_entropy', 'mem_access_type']

ransomware = set(["WannaCry", "Ryuk", "REvil", "LockBit", "Darkside", "Conti"])

ata_col_ind_to_name_map = dict(zip(range(len(strg_access_column_names)), strg_access_column_names))
mem_col_ind_to_name_map = dict(zip(range(len(mem_access_columns_names)), mem_access_columns_names))

kaggle_files_to_load = {
    'ata_read': ata_col_ind_to_name_map,
    'ata_write': ata_col_ind_to_name_map,
    'mem_read': mem_col_ind_to_name_map,
    'mem_write': mem_col_ind_to_name_map,
    'mem_readwrite': mem_col_ind_to_name_map,
    'mem_exec': mem_col_ind_to_name_map
}

# NUMBER_OF_OPERATION_LOGS_TO_LOAD = 1

kagglehub.login()

In [ ]:
HWR_PATH = "RanSMAP/dataset/original/i3-gen12/ddr4-2133-16g"


# Data should be downloaded locally to get executions names for each operation
parent_folder_path = 'RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g'
operation_names = os.listdir(parent_folder_path)
operation_paths = [os.path.join(parent_folder_path, operation) for operation in operation_names]

operations_logs_path = {}

for operation, operation_path in zip(operation_names, operation_paths):
    logs_path = [f"{operation}/{log_name}" for log_name in os.listdir(operation_path)]
    operations_logs_path[operation] = logs_path    

# Data Extraction Functions

In [ ]:
def get_datetime_min(time1, time2):
    time1_s, time1_ns = time1.unix_time_s, time1.unix_time_ns
    time2_s, time2_ns = time2.unix_time_s, time2.unix_time_ns
    
    if time1_s < time2_s:
        return time1
    elif time2_s < time1_s:
        return time2
    else:
        if time1_ns <= time2_ns:
            return time1
        return time2

def compute_relative_time(df, start_sec, start_nsec):
    """
    Compute relative time (in seconds and nanoseconds) from a start point.
    Keeps full nanosecond precision.
    """
    # Convert everything to total nanoseconds from the start point
    total_ns = (df['unix_time_s'] - start_sec) * 1_000_000_000 + (df['unix_time_ns'] - start_nsec)

    df['relative_time'] = pd.to_timedelta(total_ns, unit='ns')
    return df

In [ ]:
def read_kaggle_access_file(HWR_PATH, DATASET, operations_logs, kaggle_files_to_load):
    strg_read = None
    strg_write = None
    mem_read = None
    mem_write = None
    mem_read_write = None
    mem_exec = None
    
            
    for file, columns in kaggle_files_to_load.items():
        comb_df = pd.DataFrame()
        for log in operations_logs:
            file_path = f"{HWR_PATH}/{log}/{file}.csv"
            new_df = kagglehub.dataset_load(
                KaggleDatasetAdapter.PANDAS,
                DATASET,
                file_path,
                pandas_kwargs={"names": columns.values(), "usecols": columns.keys()}
                )
            
            comb_df = pd.concat([comb_df, new_df], ignore_index=True)
            
        match file:
            case 'ata_read':
                # strg_read = pd.concat([strg_read, new_df], ignore_index=True)
                strg_read = comb_df
            case 'ata_write':
                # strg_write = pd.concat([strg_write, new_df], ignore_index=True)
                strg_write = comb_df
            case 'mem_read':
                # mem_read = pd.concat([mem_read, new_df], ignore_index=True)
                mem_read = comb_df
            case 'mem_write':
                # mem_write = pd.concat([mem_write, new_df], ignore_index=True)
                mem_write = comb_df
            case 'mem_readwrite':
                # mem_read_write = pd.concat([mem_read_write, new_df], ignore_index=True)
                mem_read_write = comb_df
            case 'mem_exec':
                # mem_exec = pd.concat([mem_exec, new_df], ignore_index=True)
                mem_exec = comb_df
                        
    
    return strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec

def generate_combined_log_file(strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec):
    
    # strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec = read_access_file(operation_path)
    
    strg_read['action'] = 'ata_read'
    strg_write['action'] = 'ata_write'
    mem_read['action'] = 'mem_read'
    mem_write['action'] = 'mem_write'
    mem_read_write['action'] = 'mem_read_write'
    mem_exec['action'] = 'mem_exec'
    
    start_point = get_datetime_min(
                    get_datetime_min(
                        get_datetime_min(
                            get_datetime_min(
                                get_datetime_min(strg_read.loc[0, ['unix_time_s', 'unix_time_ns']], strg_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                                mem_read.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                            mem_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                        mem_read_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                    mem_exec.loc[0, ['unix_time_s', 'unix_time_ns']])


    strg_read = compute_relative_time(strg_read, start_point.unix_time_s, start_point.unix_time_ns)
    strg_write = compute_relative_time(strg_write, start_point.unix_time_s, start_point.unix_time_ns)
    mem_read = compute_relative_time(mem_read, start_point.unix_time_s, start_point.unix_time_ns)
    mem_write = compute_relative_time(mem_write, start_point.unix_time_s, start_point.unix_time_ns)
    mem_read_write = compute_relative_time(mem_read_write, start_point.unix_time_s, start_point.unix_time_ns)
    mem_exec = compute_relative_time(mem_exec, start_point.unix_time_s, start_point.unix_time_ns)
    
    combined_logs = pd.concat([strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec], ignore_index=True)
    combined_logs.drop(['unix_time_s', 'unix_time_ns'], axis=1, inplace=True)
    
    return combined_logs

In [13]:
def timedelta_to_ns(t):
    if type(t) == pd.Timedelta:
        return t.total_seconds() * 1e9 + t.nanoseconds
    else:
        return t.total_seconds() * 1e9

In [ ]:
# Temporal Evolution Features 
def temporal_evolution_features(strg_write_window_data, mem_write_window_data, mem_read_write_window_data, window_start, entropy_spike_threshold=0.9, consecutive_entropy_threshold=0.8):
    
    def compute_acceleration(signal):
        """Compute 2nd derivative (acceleration)"""
        if len(signal) < 3:
            return 0 
        first_derivative = np.gradient(signal)
        second_derivative = np.gradient(first_derivative)
        return np.mean(second_derivative)

    def count_consecutive_above_threshold(values, threshold):
        above = (values > threshold).astype(int)
        max_consecutive = 0
        current = 0
        
        for val in above:
            if val == 1:
                current += 1
                max_consecutive = max(max_consecutive, current)
            else:
                current = 0
        
        return max_consecutive

    def compute_time_to_threshold(window_data, feature, window_start, threshold):
        """Time to first event above threshold in window"""
        high_entropy = window_data[window_data[feature] > threshold]
        if len(high_entropy) == 0:
            return np.nan
        return timedelta_to_ns(high_entropy['relative_time'].iloc[0] - window_start)
    
    def get_temporal_features_for_action(window_data, feature, window_start, entropy_spike_threshold, consecutive_entropy_threshold, prefix=''):
        features = {
            f'{prefix}_high_entropy_count': (window_data[feature] > entropy_spike_threshold).sum(),
            f'{prefix}_max_entropy_jump': np.diff(window_data[feature].values).max() if len(window_data) > 1 else 0,
            f'{prefix}_time_to_high_entropy': compute_time_to_threshold(window_data, feature, window_start, entropy_spike_threshold),
            
            f'{prefix}_sustained_high_entropy_events': count_consecutive_above_threshold(
                window_data[feature].values, threshold=consecutive_entropy_threshold
            ),
            f'{prefix}_entropy_acceleration': compute_acceleration(window_data[feature].values),
        }
        return features
    
    features = {}
    
    features = features | get_temporal_features_for_action(strg_write_window_data[['relative_time', 'storage_entropy']], 'storage_entropy', window_start, 
                                                             entropy_spike_threshold, consecutive_entropy_threshold, 'strg_write')

    features = features | get_temporal_features_for_action(mem_write_window_data[['relative_time', 'mem_entropy']], 'mem_entropy', window_start, 
                                                             entropy_spike_threshold, consecutive_entropy_threshold, 'mem_write')
    
    features = features | get_temporal_features_for_action(mem_read_write_window_data[['relative_time', 'mem_entropy']], 'mem_entropy', window_start, 
                                                             entropy_spike_threshold, consecutive_entropy_threshold, 'mem_read_write')
    
    return features

# Burst Detection Features
def burst_features(strg_read_window_data, strg_write_window_data, mem_read_window_data, mem_write_window_data, mem_read_write_window_data, mem_exec_window_data, window_size_ns):
    
    def compute_instant_throughput(df, timestamp_col='relative_time', size_col='size'):
        df2 = df.copy()
        if len(df2) < 1:
            df2['instant_throughput'] = []
            return df2

        df2 = df2.sort_values(timestamp_col)
        
        start_time = df2.iloc[0][timestamp_col]
        if start_time == pd.Timedelta(0):
            start_time = pd.Timedelta(microseconds=0.001) # set start_time to 1ns

        grouped = df2.groupby(timestamp_col, sort=True)[size_col].sum().reset_index().rename(columns={size_col: 'size_sum'})

        grouped['time_diff_ns'] = grouped[timestamp_col].diff().apply(lambda x: timedelta_to_ns(start_time) if pd.isnull(x) else timedelta_to_ns(x))
        grouped['throughput'] = grouped['size_sum'] / grouped['time_diff_ns']

        mapping = dict(zip(grouped[timestamp_col], grouped['throughput']))
        df2["instant_throughput"] = df2[timestamp_col].map(mapping)
        
        return df2
    
    def compute_burst_clustering(window_data, quantile_threshold = 0.75):
        """Measure temporal clustering of high-throughput events"""
        if len(window_data) < 2:
            return 0
        
        threshold = window_data['instant_throughput'].quantile(quantile_threshold)
        burst_times = window_data[window_data['instant_throughput'] > threshold]['relative_time'].values
        
        if len(burst_times) < 2:
            return 0
        
        inter_burst_intervals = np.diff(burst_times)
        
        return np.mean(inter_burst_intervals).astype(int)
    
    def get_burst_features_for_action(window_data, prefix, size_col, window_size_ns):
        window_data_w_throughput = compute_instant_throughput(window_data, size_col=size_col)
        features = {
            f'{prefix}_peak_instant_throughput': window_data_w_throughput['instant_throughput'].max(),
                        
            f'{prefix}_burst_intensity': (window_data_w_throughput['instant_throughput'].max() / (window_data_w_throughput['instant_throughput'].median() + 1e-10)),
            
            f'{prefix}_micro_burst_count': (window_data_w_throughput['instant_throughput'] > 
                                window_data_w_throughput['instant_throughput'].mean() + 2 * window_data_w_throughput['instant_throughput'].std()).sum(),
            
            f'{prefix}_burst_clustering': compute_burst_clustering(window_data_w_throughput),
            
            f'{prefix}_event_rate': len(window_data_w_throughput) / window_size_ns,
        }
        
        return features
    
    features = {}
    
    features = features | get_burst_features_for_action(strg_read_window_data, 'strg_read', 'storage_size', window_size_ns)
    features = features | get_burst_features_for_action(strg_write_window_data, 'strg_write', 'storage_size', window_size_ns)
    features = features | get_burst_features_for_action(mem_read_window_data, 'mem_read', 'mem_size', window_size_ns)
    features = features | get_burst_features_for_action(mem_write_window_data, 'mem_write', 'mem_size', window_size_ns)
    features = features | get_burst_features_for_action(mem_read_write_window_data, 'mem_read_write', 'mem_size', window_size_ns)
    features = features | get_burst_features_for_action(mem_exec_window_data, 'mem_exec', 'mem_size', window_size_ns)
    
    return features

# Address Spatial Access Features
def spatial_access_patterns(strg_read_window_data, strg_write_window_data, mem_read_window_data, mem_write_window_data, mem_read_write_window_data, mem_exec_window_data):
    """
    Analyze LBA and GPA access sequences from raw events
    """
    
    def compute_sequential_ratio(addr_sequence):
        """Fraction of address accesses that are sequential"""
        if len(addr_sequence) < 2:
            return 0
        
        diffs = np.abs(np.diff(addr_sequence))
        sequential = (diffs == 1).sum()
        return sequential / len(diffs)

    def compute_access_direction_ratio(addr_sequence):
        if len(addr_sequence) < 2:
            return 1.0
        
        diffs = np.diff(addr_sequence)
        forward = (diffs > 0).sum()
        backward = (diffs < 0).sum()
        
        return forward / (backward + 1e-10)

    def get_spatial_access_features_for_action(addr_sequence, prefix):

        if prefix in ['strg_read', 'strg_write']:
            addr_name = 'lba'
        elif prefix in ['mem_read', 'mem_write', 'mem_read_write', 'mem_exec']:
            addr_name = 'gpa'
        else:
            addr_name = 'addr'
        features = { 
            f'{prefix}_seq_addr_access_ratio': compute_sequential_ratio(addr_sequence),
            
            f'{prefix}_avg_{addr_name}_jump': np.mean(np.abs(np.diff(addr_sequence))) if len(addr_sequence) > 1 else 0,
            f'{prefix}_max_{addr_name}_jump': np.max(np.abs(np.diff(addr_sequence))) if len(addr_sequence) > 1 else 0,
            
            f'{prefix}_access_locality': len(np.unique(addr_sequence)) / len(addr_sequence) if len(addr_sequence) > 0 else 0,
            
            f'{prefix}_frwrd_bckwrd_ratio': compute_access_direction_ratio(addr_sequence),
            
            f'{prefix}_{addr_name}_range': addr_sequence.max() - addr_sequence.min() if len(addr_sequence) > 0 else 0,
        }
        return features
    
    
    strg_read_addr_sequence = strg_read_window_data['lba'].values
    strg_write_addr_sequence = strg_write_window_data['lba'].values
    mem_read_addr_sequence = mem_read_window_data['gpa'].values
    mem_write_addr_sequence = mem_write_window_data['gpa'].values
    mem_read_write_addr_sequence = mem_read_write_window_data['gpa'].values
    mem_exec_addr_sequence = mem_exec_window_data['gpa'].values
    
    features = {}
    features = features | get_spatial_access_features_for_action(strg_read_addr_sequence, 'strg_read')
    features = features | get_spatial_access_features_for_action(strg_write_addr_sequence, 'strg_write')
    features = features | get_spatial_access_features_for_action(mem_read_addr_sequence, 'mem_read')
    features = features | get_spatial_access_features_for_action(mem_write_addr_sequence, 'mem_write')
    features = features | get_spatial_access_features_for_action(mem_read_write_addr_sequence, 'mem_read_write')
    features = features | get_spatial_access_features_for_action(mem_exec_addr_sequence, 'mem_exec')
        
    return features

In [ ]:
def ransmap_preprocessing(df: pd.DataFrame, is_ransomware: bool, operation: str, window_s: int, epsilon_s: int = 1) -> pd.DataFrame:
    '''
    window_s: Window size in seconds
    epsilon_s: Step size in seconds
    '''
    
    data_rows = []
    window_delta = datetime.timedelta(seconds=window_s)
    epsilon_delta = datetime.timedelta(seconds=epsilon_s)
    
    last_start_possible = max(df['relative_time']) - window_delta
        
    start_time = min(df['relative_time'])
    
    strg_read = df[df['action'] == 'ata_read'][["relative_time", "lba", "storage_size"]]
    strg_write = df[df['action'] == 'ata_write'][["relative_time", "lba", "storage_size", "storage_entropy"]]
    mem_read = df[df['action'] == 'mem_read'][["relative_time", "gpa", 'mem_size', 'mem_entropy', 'mem_access_type']]
    mem_write = df[df['action'] == 'mem_write'][["relative_time", "gpa", 'mem_size', 'mem_entropy', 'mem_access_type']]
    mem_read_write = df[df['action'] == 'mem_read_write'][["relative_time", "gpa", 'mem_size', 'mem_entropy', 'mem_access_type']]
    mem_exec = df[df['action'] == 'mem_exec'][["relative_time", "gpa", 'mem_size', 'mem_entropy', 'mem_access_type']]
    
    while start_time <= last_start_possible:
        try:            
            end_time = start_time + window_delta
            
            d_ata_read = strg_read[(start_time <= strg_read['relative_time']) & (strg_read['relative_time'] <= end_time)]
            d_ata_write = strg_write[(start_time <= strg_write['relative_time']) & (strg_write['relative_time'] <= end_time)]
            d_mem_read = mem_read[(start_time <= mem_read['relative_time']) & (mem_read['relative_time'] <= end_time)]
            d_mem_write = mem_write[(start_time <= mem_write['relative_time']) & (mem_write['relative_time'] <= end_time)]
            d_mem_read_write = mem_read_write[(start_time <= mem_read_write['relative_time']) & (mem_read_write['relative_time'] <= end_time)]
            d_mem_exec = mem_exec[(start_time <= mem_exec['relative_time']) & (mem_exec['relative_time'] <= end_time)]

            features = {}
            
            features = features | {"start": start_time, "end": end_time}
            
            features = features | {
                "avg_ata_read": d_ata_read['storage_size'].sum() / window_s,
                "avg_ata_write": d_ata_write['storage_size'].sum() / window_s,
                "lba_read_var": d_ata_read['lba'].var(),
                "lba_write_var": d_ata_write['lba'].var(),
                "avg_storage_entropy": d_ata_write['storage_entropy'].mean()
            }
            
            features = features | {
                "avg_entropy_mem_write": d_mem_write['mem_entropy'].mean(),
                "avg_entropy_mem_read_write": d_mem_read_write['mem_entropy'].mean()
            }

            features = features | {
                "num_4KB_pages_mem_read": len(d_mem_read[d_mem_read['mem_access_type'] == 1]),
                "num_4KB_pages_mem_write": len(d_mem_write[d_mem_write['mem_access_type'] == 1]),
                "num_4KB_pages_mem_read_write": len(d_mem_read_write[d_mem_read_write['mem_access_type'] == 1]),
                "num_4KB_pages_mem_exec": len(d_mem_exec[d_mem_exec['mem_access_type'] == 1]),
                
                "num_2MB_pages_mem_read": len(d_mem_read[d_mem_read['mem_access_type'] == 2]),
                "num_2MB_pages_mem_write": len(d_mem_write[d_mem_write['mem_access_type'] == 2]),
                "num_2MB_pages_mem_read_write": len(d_mem_read_write[d_mem_read_write['mem_access_type'] == 2]),
                "num_2MB_pages_mem_exec": len(d_mem_exec[d_mem_exec['mem_access_type'] == 2]),
                
                "num_MMIO_pages_mem_read": len(d_mem_read[d_mem_read['mem_access_type'] == 4]),
                "num_MMIO_pages_mem_write": len(d_mem_write[d_mem_write['mem_access_type'] == 4]),
                "num_MMIO_pages_mem_read_write": len(d_mem_read_write[d_mem_read_write['mem_access_type'] == 4]),
                "num_MMIO_pages_mem_exec": len(d_mem_exec[d_mem_exec['mem_access_type'] == 4]),
            }
            
            features = features | {
                "gpa_mem_read_var": d_mem_read['gpa'].var(),
                "gpa_mem_write_var": d_mem_write['gpa'].var(),
                "gpa_mem_read_write_var": d_mem_read_write['gpa'].var(),
                "gpa_mem_exec_var": d_mem_exec['gpa'].var()
            }
            
            features = features | temporal_evolution_features(d_ata_write, d_mem_write, d_mem_read_write, start_time)
            features = features | burst_features(d_ata_read, d_ata_write, d_mem_read, d_mem_write, d_mem_read_write, d_mem_exec, window_s*1e9)
            features = features | spatial_access_patterns(d_ata_read, d_ata_write, d_mem_read, d_mem_write, d_mem_read_write, d_mem_exec)
        
        except:
            print("Error occurred at start time: ", start_time)
            return None
        # preprocessed_df.loc[len(preprocessed_df)] = new
        data_rows.append(features)
        
        start_time += epsilon_delta
    
    preprocessed_df = pd.DataFrame(data_rows)
    preprocessed_df['is_ransomware'] = is_ransomware
    preprocessed_df['operation'] = operation
    return preprocessed_df

# Actual Extraction

In [16]:
def generate_dataset(HWR_PATH, DATASET, operations_logs_to_load, kaggle_files_to_load, window_s: int, epsilon_s: int, ransomware):
    final_dataset = pd.DataFrame()
    for operation, operation_logs in operations_logs_to_load.items():
        for operation_log_cnt, log in enumerate(operation_logs):
            
            print(f"Reading {operation} log {operation_log_cnt+1} ({log}) from Kaggle")
            strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec = read_kaggle_access_file(HWR_PATH, DATASET, [log], kaggle_files_to_load)
            print(f"Combining {operation} log {operation_log_cnt+1} files")
            combined_log_files = generate_combined_log_file(strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec)
            print(f"Creating features using {operation} log {operation_log_cnt+1} files")
            feature_dataset = ransmap_preprocessing(combined_log_files, operation in ransomware, operation, window_s, epsilon_s)
            print(f"Finished Creating feature dataset for {operation} log {operation_log_cnt+1}\n")
            feature_dataset['operation_log_cnt'] = operation_log_cnt+1
            
            final_dataset = pd.concat([final_dataset, feature_dataset], ignore_index=True)
    
    final_dataset['start_s'] = final_dataset['start'].apply(lambda x: x.seconds)
    final_dataset['end_s'] = final_dataset['end'].apply(lambda x: x.seconds)
    return final_dataset

In [ ]:
df = generate_dataset(HWR_PATH, DATASET, operations_logs_path, kaggle_files_to_load, 10, 1, ransomware)

In [3]:
df.head()

,start,end,avg_ata_read,avg_ata_write,lba_read_var,lba_write_var,avg_storage_entropy,avg_entropy_mem_write,avg_entropy_mem_read_write,num_4KB_pages_mem_read,...,mem_exec_avg_gpa_jump,mem_exec_max_gpa_jump,mem_exec_access_locality,mem_exec_frwrd_bckwrd_ratio,mem_exec_gpa_range,is_ransomware,operation,operation_log_cnt,start_s,end_s
0,0 days 00:00:00,0 days 00:00:10,420300.8,78233.6,8.750598e+14,2.836417e+15,0.742203,0.049471,0.753775,9,...,5.529372e+09,1.951698e+10,1.0,0.959677,1.956886e+10,False,AESCrypt,1,0,10
1,0 days 00:00:01,0 days 00:00:11,420300.8,78233.6,8.750598e+14,2.836417e+15,0.742203,0.011468,0.595073,1,...,4.865936e+09,1.218004e+10,1.0,0.909091,1.937828e+10,False,AESCrypt,1,1,11
2,0 days 00:00:02,0 days 00:00:12,1121177.6,79052.8,4.981796e+14,2.843206e+15,0.740121,0.004692,NaN,0,...,2.166676e+09,7.452113e+09,1.0,0.333333,7.554102e+09,False,AESCrypt,1,2,12
3,0 days 00:00:03,0 days 00:00:13,1362073.6,6144.0,5.017219e+14,4.499802e+14,0.399652,0.008459,0.637976,0,...,2.758968e+09,7.482259e+09,1.0,1.000000,7.887104e+09,False,AESCrypt,1,3,13
4,0 days 00:00:04,0 days 00:00:14,1407027.2,4505.6,4.985177e+14,2.045721e+13,0.362840,0.008541,0.637976,0,...,2.758968e+09,7.482259e+09,1.0,1.000000,7.887104e+09,False,AESCrypt,1,4,14


In [ ]:
with open(os.path.join("processed_dataset", "final_dataset_new_features_all_logs_10s_window.pkl"), "wb") as file:
    pickle.dump(df, file)